# MoodSense AI v2 - Model Training Pipeline

This notebook demonstrates the complete model training pipeline for MoodSense AI.

## Overview
1. Data Preparation
2. Feature Engineering (TF-IDF + Embeddings)
3. Model Training (Logistic Regression, Naive Bayes, LightGBM)
4. Evaluation & Comparison
5. Model Selection & Saving

In [ ]:
# Install dependencies (if running on Kaggle/Colab)
# !pip install -q fastapi uvicorn pydantic numpy pandas scikit-learn lightgbm sentence-transformers spacy transformers torch gradio httpx python-dotenv structlog mlflow
# !python -m spacy download en_core_web_sm

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from app.core.config import get_settings
from app.core.constants import SAMPLE_TRAINING_DATA, MOOD_DESCRIPTIONS
from app.core.logging import get_logger, setup_logging
from app.models.trainer import ModelTrainer
from app.services.preprocessing import TextPreprocessor
from app.services.embeddings import EmbeddingService

setup_logging()
logger = get_logger(__name__)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Exploration

In [ ]:
# Load sample data
texts, labels = zip(*SAMPLE_TRAINING_DATA)

# Create DataFrame
df = pd.DataFrame({'text': texts, 'mood': labels})

print(f"Total samples: {len(df)}")
print(f"Mood distribution:")
print(df['mood'].value_counts())

In [ ]:
# Visualize mood distribution
fig, ax = plt.subplots(figsize=(12, 6))
mood_counts = df['mood'].value_counts()
colors = [MOOD_DESCRIPTIONS.get(m, '#gray') for m in mood_counts.index]

mood_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Mood Distribution in Training Data', fontsize=14, fontweight='bold')
ax.set_xlabel('Mood', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 2. Text Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = TextPreprocessor(use_spacy=False)

# Preprocess sample texts
sample_texts = df['text'].head(5).tolist()
processed = [preprocessor.preprocess(t) for t in sample_texts]

print("Original vs Processed:")
for orig, proc in zip(sample_texts, processed):
    print(f"\nOriginal: {orig}")
    print(f"Processed: {proc}")

## 3. Feature Engineering

In [ ]:
# Initialize embedding service
embedding_service = EmbeddingService(
    embedding_model='all-MiniLM-L6-v2',
    use_tfidf=True,
    use_sentence_embeddings=True
)

# Preprocess all texts
processed_texts = [preprocessor.preprocess(t) for t in texts]

# Fit TF-IDF
embedding_service.fit_tfidf(processed_texts)

# Generate combined embeddings
X = embedding_service.get_combined_embeddings(processed_texts)

print(f"Embedding shape: {X.shape}")
print(f"TF-IDF features: {embedding_service.tfidf_vectorizer.max_features}")
print(f"Sentence embedding dim: {embedding_service._embedding_dim}")

## 4. Model Training

In [ ]:
# Initialize trainer
trainer = ModelTrainer(test_size=0.2, random_state=42, use_mlflow=False)

# Prepare data
trainer.X_train = X_train
trainer.X_test = X_test
trainer.y_train = y_train
trainer.y_test = y_test

# Create label mappings
from sklearn.model_selection import train_test_split
unique_labels = sorted(set(labels))
label_map = {label: idx for idx, label in enumerate(unique_labels)}
reverse_label_map = {idx: label for label, idx in label_map.items()}

y = np.array([label_map[label] for label in labels])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

trainer.X_train = X_train
trainer.X_test = X_test
trainer.y_train = y_train
trainer.y_test = y_test
trainer.label_map = label_map
trainer.reverse_label_map = reverse_label_map

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
# Train Logistic Regression
lr_result = trainer.train_logistic_regression()
print(f"\nLogistic Regression Results:")
print(f"  Accuracy: {lr_result.metrics.accuracy:.4f}")
print(f"  F1 Score: {lr_result.metrics.f1_score:.4f}")
print(f"  Best params: {lr_result.best_params}")

In [ ]:
# Train LightGBM
lgb_result = trainer.train_lightgbm()
print(f"\nLightGBM Results:")
print(f"  Accuracy: {lgb_result.metrics.accuracy:.4f}")
print(f"  F1 Score: {lgb_result.metrics.f1_score:.4f}")
print(f"  Best params: {lgb_result.best_params}")

## 5. Model Comparison

In [ ]:
# Compare models
results = [lr_result, lgb_result]

comparison_data = []
for r in results:
    comparison_data.append({
        'Model': r.model_name,
        'Accuracy': r.metrics.accuracy,
        'Precision': r.metrics.precision,
        'Recall': r.metrics.recall,
        'F1 Score': r.metrics.f1_score
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nModel Comparison:")
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(comparison_df))
width = 0.2

ax.bar(x - 1.5*width, comparison_df['Accuracy'], width, label='Accuracy')
ax.bar(x - 0.5*width, comparison_df['Precision'], width, label='Precision')
ax.bar(x + 0.5*width, comparison_df['Recall'], width, label='Recall')
ax.bar(x + 1.5*width, comparison_df['F1 Score'], width, label='F1 Score')

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'])
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 6. Save Best Model

In [ ]:
# Select best model
best = trainer.select_best_model(results, metric='f1_score')
print(f"\nBest Model: {best.model_name}")
print(f"F1 Score: {best.metrics.f1_score:.4f}")

# Save model
import os
os.makedirs('../models', exist_ok=True)
save_path = trainer.save_model(best, '../models')
print(f"\nModel saved to: {save_path}")

## 7. Test Predictions

In [ ]:
from app.models.predictor import MoodPredictor

# Load predictor
predictor = MoodPredictor(model_path='../models/mood_classifier_v2.pkl')

# Test predictions
test_texts = [
    "I am so happy today!",
    "I feel really sad and down",
    "This makes me angry!",
    "I'm worried about tomorrow",
]

for text in test_texts:
    prediction = predictor.predict(text)
    print(f"\nText: {text}")
    print(f"Mood: {prediction.mood} (confidence: {prediction.confidence:.2%})")

## Conclusion

This notebook demonstrated:
1. Data preprocessing and exploration
2. Feature engineering with TF-IDF and sentence embeddings
3. Training multiple classification models
4. Model evaluation and selection
5. Saving and testing the best model

Next steps:
- Deploy the model using the FastAPI service
- Use the Gradio UI for interactive predictions
- Monitor performance with MLflow